In [1]:
import pandas as pd
import numpy as np

movies = pd.read_csv(
    r"c:\xampp\perl\vendor\lib\auto\Devel\vscode\TMDB_data\cleaned_movies.csv"
)

genres = pd.read_csv(
    r"c:\xampp\perl\vendor\lib\auto\Devel\vscode\TMDB_data\cleaned_genres.csv"
)

movie_genres = pd.read_csv(
    r"c:\xampp\perl\vendor\lib\auto\Devel\vscode\TMDB_data\cleaned_movie_genres.csv"
)

cast = pd.read_csv(
    r"c:\xampp\perl\vendor\lib\auto\Devel\vscode\TMDB_data\cleaned_cast.csv"
)

crew = pd.read_csv(
    r"c:\xampp\perl\vendor\lib\auto\Devel\vscode\TMDB_data\cleaned_crew.csv"
)

movie_keywords = pd.read_csv(
    r"c:\xampp\perl\vendor\lib\auto\Devel\vscode\TMDB_data\cleaned_movie_keywords.csv"
)

print("All cleaned datasets loaded successfully!")

All cleaned datasets loaded successfully!


This provides a high-level overview of the size and diversity of the TMDB dataset.

In [2]:
print("Total Unique Movies:", movies["movie_id"].nunique())
print("Total Genres:", genres["genre_id"].nunique())
print("Total Unique Cast Members:", cast["person_id"].nunique())
print("Total Unique Crew Members:", crew["person_id"].nunique())

Total Unique Movies: 2503
Total Genres: 19
Total Unique Cast Members: 11300
Total Unique Crew Members: 2848


Movies with missing or zero budget/revenue should generally be excluded from profitability and ROI calculations because those calculations would not be reliable.


In [3]:
q2_result = movies[
    (movies["budget"].isna()) |
    (movies["revenue"].isna()) |
    (movies["budget"] == 0) |
    (movies["revenue"] == 0)
][["movie_id", "title", "budget", "revenue"]]

q2_result

,movie_id,title,budget,revenue
324,791373,Zack Snyder's Justice League,70000000.0,0.0
340,405774,Bird Box,19800000.0,0.0
403,664413,365 Days,0.0,9458590.0
468,466282,To All the Boys I've Loved Before,0.0,0.0
554,766507,Prey,65000000.0,0.0
...,...,...,...,...
2469,755566,Day Shift,100000000.0,0.0
2471,73861,A Serbian Film,0.0,0.0
2480,11104,Chungking Express,160000.0,0.0
2481,500840,I'm Thinking of Ending Things,0.0,0.0


In [4]:
valid_financial_movies = movies[
    (movies["budget"] > 0)
    & (movies["revenue"] > 0)
].copy()

In [5]:
q3_result = movies[
    (movies["budget"] > 0)
    & (movies["revenue"] > 0)
].copy()

q3_result["profit"] = (
    q3_result["revenue"] - q3_result["budget"]
)

q3_result["roi"] = (
    q3_result["profit"] / q3_result["budget"]
) * 100

q3_result[
    ["movie_id", "title", "budget", "revenue", "profit", "roi"]
]

,movie_id,title,budget,revenue,profit,roi
0,157336,Interstellar,1.650000e+08,7.466067e+08,5.816067e+08,352.488913
1,27205,Inception,1.600000e+08,8.390306e+08,6.790306e+08,424.394144
2,24428,The Avengers,2.200000e+08,1.518816e+09,1.298816e+09,590.370689
3,155,The Dark Knight,1.850000e+08,1.004558e+09,8.195584e+08,443.004564
4,19995,Avatar,2.370000e+08,2.923706e+09,2.686706e+09,1133.631235
...,...,...,...,...,...,...
2498,19901,Daybreakers,2.000000e+07,5.141719e+07,3.141719e+07,157.085940
2499,1368167,Date Night,9.321480e+07,8.759568e+07,-5.619119e+06,-6.028140
2500,1368168,Transformers,1.100988e+08,2.747467e+08,1.646479e+08,149.545660
2501,1368169,The SpongeBob Movie: Sponge Out of Water,3.928488e+07,5.706575e+08,5.313726e+08,1352.613653


In [6]:
q4_result = (
    q3_result[
        q3_result["budget"] >= 1_000_000
    ]
    .sort_values("roi", ascending=False)
    .head(15)
)

q4_result[
    ["title", "budget", "revenue", "profit", "roi"]
]

,title,budget,revenue,profit,roi
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,12400.282100
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,12324.256142
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,169781400.0,12127.242857
467,Rocky,1000000.0,117253345.0,116253345.0,11625.334500
1227,Gone with the Wind,4000000.0,402352579.0,398352579.0,9958.814475
716,The Jungle Book,4000000.0,378000000.0,374000000.0,9350.000000
635,Cinderella,2900000.0,263600000.0,260700000.0,8989.655172
372,Saw,1200000.0,104045735.0,102845735.0,8570.477917
719,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,8316.666667
263,E.T. the Extra-Terrestrial,10500000.0,797307407.0,786807407.0,7493.403876


In [7]:
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

movies["release_year"] = (
    movies["release_date"].dt.year
)

q5_result = (
    movies.groupby("release_year")
    .size()
    .reset_index(name="movie_count")
    .sort_values("movie_count", ascending=False)
)

q5_result

,release_year,movie_count
78,2016,121
75,2013,110
79,2017,109
76,2014,105
80,2018,102
...,...,...
0,1921,1
11,1946,1
32,1970,1
34,1972,1


In [8]:
highest_year = q5_result.iloc[0]

print("Highest movie output year:", int(highest_year["release_year"]))
print("Number of movies:", highest_year["movie_count"])

Highest movie output year: 2016
Number of movies: 121


In [9]:
runtime = movies["runtime"].dropna()

Q1 = runtime.quantile(0.25)
Q3 = runtime.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

q6_result = movies[
    (movies["runtime"] < lower_bound)
    | (movies["runtime"] > upper_bound)
][["movie_id", "title", "runtime"]]

q6_result

,movie_id,title,runtime
0,157336,Interstellar,169
16,299534,Avengers: Endgame,181
18,120,The Lord of the Rings: The Fellowship of the Ring,179
19,597,Titanic,194
20,122,The Lord of the Rings: The Return of the King,201
21,106646,The Wolf of Wall Street,180
28,121,The Lord of the Rings: The Two Towers,179
38,238,The Godfather,175
76,49051,The Hobbit: An Unexpected Journey,169
79,497,The Green Mile,189


In [10]:
movie_genre_details = movie_genres.merge(
    genres,
    on="genre_id",
    how="left"
)

primary_genres = (
    movie_genre_details
    .drop_duplicates(
        subset="movie_id",
        keep="first"
    )
)

In [11]:
movies_with_primary_genre = movies.merge(
    primary_genres[
        ["movie_id", "genre_name"]
    ],
    on="movie_id",
    how="left"
)

In [12]:
q7_result = (
    movies_with_primary_genre
    .groupby("genre_name")["popularity"]
    .mean()
    .sort_values(ascending=False)
    .reset_index(name="average_popularity")
)

q7_result

,genre_name,average_popularity
0,Science Fiction,21.261836
1,Music,18.324340
2,Animation,16.811212
3,Family,16.710635
4,Action,16.385188
5,War,15.470317
6,Horror,14.851835
7,Adventure,14.785510
8,Fantasy,13.022920
9,Mystery,12.427131


In [13]:
actor_movies = cast.merge(
    movies[
        ["movie_id", "vote_average"]
    ],
    on="movie_id",
    how="left"
)

In [14]:
q8_result = (
    actor_movies
    .groupby(
        ["person_id", "actor_name"]
    )
    .agg(
        movie_count=(
            "movie_id",
            "nunique"
        ),
        average_movie_rating=(
            "vote_average",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "movie_count",
        ascending=False
    )
    .head(10)
)

q8_result

,person_id,actor_name,movie_count,average_movie_rating
679,2231,Samuel L. Jackson,39,6.938692
106,287,Brad Pitt,38,7.257289
127,380,Robert De Niro,38,7.228316
37,85,Johnny Depp,37,6.856973
13,31,Tom Hanks,33,7.289758
1272,5293,Willem Dafoe,32,7.114781
413,1245,Scarlett Johansson,32,7.056344
2674,13240,Mark Wahlberg,32,6.630063
161,500,Tom Cruise,31,7.040613
64,192,Morgan Freeman,31,6.963742


In [15]:
q9_result = movies[
    movies.duplicated(
        subset="title",
        keep=False
    )
].sort_values(
    ["title", "release_year"]
)

q9_result[
    [
        "movie_id",
        "title",
        "release_year"
    ]
]

,movie_id,title,release_year
878,377,A Nightmare on Elm Street,1984
1928,23437,A Nightmare on Elm Street,2010
261,812,Aladdin,1992
339,420817,Aladdin,2019
792,12092,Alice in Wonderland,1951
...,...,...,...
1737,60935,The Thing,2011
744,861,Total Recall,1990
860,64635,Total Recall,2012
2500,1368168,Transformers,1987


In [16]:
high_popularity = movies["popularity"].quantile(0.90)
low_popularity = movies["popularity"].quantile(0.10)

high_votes = movies["vote_count"].quantile(0.90)
low_votes = movies["vote_count"].quantile(0.10)

In [17]:
q10_result = movies[
    (
        (movies["popularity"] >= high_popularity)
        & (movies["vote_count"] <= low_votes)
    )
    |
    (
        (movies["popularity"] <= low_popularity)
        & (movies["vote_count"] >= high_votes)
    )
][
    [
        "movie_id",
        "title",
        "popularity",
        "vote_count",
        "vote_average"
    ]
]

q10_result

,movie_id,title,popularity,vote_count,vote_average
205,49538,X-Men: First Class,1.2614,13680,7.303
244,771,Home Alone,1.8114,12658,7.469
2248,1083381,Backrooms,258.4743,2483,7.100
2265,1084199,Companion,23.1407,2466,7.009
2295,1304313,Lee Cronin's The Mummy,79.4601,2430,7.943
2306,539972,Kraven the Hunter,26.1998,2419,6.400
2388,1275779,Disclosure Day,394.3346,2328,7.427
2441,802,Lolita,27.5391,2271,7.300
2452,931285,Mortal Kombat II,99.5231,2260,7.906
2457,1100782,Smile 2,23.7112,2256,6.585
